# Stanford Dogs 120 — CAM & Grad-CAM Object Localization

이 노트북은 `stanford_dogs_cam_gradcam.py` 스크립트의 로직을 노트북 형태로 재구성한 것입니다.
ResNet50 기반 분류기를 학습한 뒤, **CAM**과 **Grad-CAM** 두 가지 방식으로 class activation map을 얻고,
이를 바운딩 박스로 변환하여 실제 GT(Ground Truth) 바운딩 박스와의 **IoU**를 계산해 두 방법의
object localization 성능을 비교 분석합니다.

## 프로젝트 루브릭
1. **인식결과의 시각화 및 성능 분석** — CAM/Grad-CAM 각각에 대해 원본이미지 합성, 바운딩박스, IoU 계산 과정을 통해 두 방법의 localization 성능을 비교분석한다.
2. **설명 가능한 CAM 획득** — CAM과 Grad-CAM 방식의 class activation map이 정상적으로 얻어지고, 시각화 시 물체의 주요 특징 위치를 잘 반영한다.
3. **기본 모델 구성과 학습** — ResNet50 + GAP + Dense(FC) 레이어로 구성된 CAM 모델의 학습이 안정적으로 수렴한다.

## 데이터 준비
공식 페이지(http://vision.stanford.edu/aditya86/ImageNetDogs/main.html) 에서 압축 파일을 내려받아
아래와 같은 구조로 압축을 해제한 뒤 `DATA_ROOT` 경로만 맞춰주면 됩니다.

```
stanford-dogs/
  Images/n02085620-Chihuahua/*.jpg
  Annotation/n02085620-Chihuahua/*.xml
  lists/train_list.mat, lists/test_list.mat
```


In [ ]:
# (최초 1회) 데이터 다운로드 예시 — 이미 데이터가 있다면 건너뛰어도 됩니다.
!mkdir -p ~/work/class_activation_map && cd ~/work/class_activation_map && \
wget http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar && \
wget http://vision.stanford.edu/aditya86/ImageNetDogs/annotation.tar && \
wget http://vision.stanford.edu/aditya86/ImageNetDogs/lists.tar && \
mkdir -p stanford-dogs/lists && \
tar -xf images.tar -C stanford-dogs && tar -xf annotation.tar -C stanford-dogs && \
tar -xf lists.tar -C stanford-dogs/lists
print("데이터가 이미 준비되어 있다면 이 셀은 건너뛰고 DATA_ROOT만 지정하세요.")

--2026-08-27 13:22:31--  http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar
Resolving vision.stanford.edu (vision.stanford.edu)... 171.64.68.10
Connecting to vision.stanford.edu (vision.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 793579520 (757M) [application/x-tar]
Saving to: ‘images.tar.3’

images.tar.3         81%[===============>    ] 614.69M  7.85MB/s    eta 12s    

In [ ]:
import os
import random
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.io import loadmat
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

SEED = 42
IMG_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# 데이터 루트 경로 (환경에 맞게 수정)
DATA_ROOT = os.path.expanduser("~/work/class_activation_map/stanford-dogs")
LIMIT_PER_CLASS = 8   # 클래스당 사용할 이미지 수 (실습용으로 소규모 서브셋)
EPOCHS = 5            # FC 레이어만 학습하는 epoch 수
BATCH_SIZE = 16

## 1. 데이터셋 정의

`.mat` 파일에서 train/test 이미지 목록을 읽고, 각 이미지에 대응하는 XML 어노테이션에서
바운딩 박스를 파싱하는 `StanfordDogs` Dataset을 정의합니다. 이 데이터셋은
`(image, label, box, original_size, path)` 를 반환합니다.

In [ ]:
def read_split(root, split):
    '''lists/{split}_list.mat 에서 상대 이미지 경로 리스트를 읽어온다.'''
    mat = loadmat(Path(root) / "lists" / f"{split}_list.mat")
    key = "file_list" if "file_list" in mat else f"{split}_list"   # ← 이 줄이 수정 포인트
    raw = mat[key]
    paths = []
    for item in raw[:, 0]:
        value = item[0] if isinstance(item, np.ndarray) else item
        if isinstance(value, bytes):
            value = value.decode("utf-8")
        paths.append(str(value))
    return paths


def box_from_xml(path):
    '''Annotation XML에서 bndbox를 읽어 [xmin, ymin, xmax, ymax] (0-based)로 반환.'''
    root = ET.parse(path).getroot()
    box = root.find("object/bndbox")
    if box is None:
        raise ValueError(f"bounding box not found: {path}")
    return [int(box.findtext("xmin")) - 1, int(box.findtext("ymin")) - 1,
            int(box.findtext("xmax")) - 1, int(box.findtext("ymax")) - 1]

In [ ]:
class StanfordDogs(Dataset):
    '''Stanford Dogs 120 이미지 + 클래스 + GT 바운딩박스를 함께 반환하는 Dataset.'''

    def __init__(self, root, split="train", transform=None, limit_per_class=8):
        self.root = Path(root)
        self.transform = transform
        names = read_split(root, split)
        self.items = []
        classes = sorted({Path(p).parent.name for p in names})
        self.class_to_idx = {name: i for i, name in enumerate(classes)}
        self.classes = classes
        counts = {c: 0 for c in classes}
        for rel in names:
            breed = Path(rel).parent.name
            if counts[breed] >= limit_per_class:
                continue
            image = self.root / "Images" / rel

            stem = rel.rsplit(".", 1)[0]
            xml_no_ext = self.root / "Annotation" / stem
            xml_with_ext = xml_no_ext.with_suffix(".xml")
            if xml_no_ext.exists():
                xml_path = xml_no_ext          # ← 확장자 없는 경로를 우선 사용
            elif xml_with_ext.exists():
                xml_path = xml_with_ext
            else:
                xml_path = None
            if not image.exists():
                continue
            if xml_path is None:
                continue

            self.items.append((image, xml_path, self.class_to_idx[breed]))
            counts[breed] += 1
        if not self.items:
            raise FileNotFoundError("Images/Annotation/lists 경로와 압축 해제 상태를 확인하세요.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        image_path, xml_path, target = self.items[idx]
        image = Image.open(image_path).convert("RGB")
        original_size = image.size
        box = box_from_xml(xml_path)
        if self.transform:
            image = self.transform(image)
        return image, target, box, original_size, str(image_path)

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = StanfordDogs(DATA_ROOT, "train", train_tf, LIMIT_PER_CLASS)
test_dataset = StanfordDogs(DATA_ROOT, "test", eval_tf, LIMIT_PER_CLASS)

print(f"클래스 수: {len(train_dataset.classes)}")
print(f"학습 샘플 수: {len(train_dataset)}")
print(f"테스트 샘플 수: {len(test_dataset)}")

## 2. 샘플 시각화 (원본 이미지 + GT 바운딩 박스)

CAM/Grad-CAM 비교를 위해 우선 원본 이미지와 정답 바운딩 박스가 잘 로드되는지 확인합니다.

In [ ]:
def unnormalize(img_tensor):
    '''정규화된 텐서를 [0, 255] uint8 RGB 이미지(H, W, C)로 되돌린다.'''
    mean = np.array(MEAN)
    std = np.array(STD)
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return np.uint8(255 * img)


def show_sample_with_box(dataset, idx):
    image, label, box, size, path = dataset[idx]
    img_np = unnormalize(image)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img_np)
    x0, y0, x1, y1 = map_box(box, size)
    rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2,
                              edgecolor="lime", facecolor="none")
    ax.add_patch(rect)
    ax.set_title(f"class: {dataset.classes[label]}")
    ax.axis("off")
    plt.show()

In [ ]:
def map_box(box, original_size):
    '''원본 이미지 크기 기준 box 좌표를 IMG_SIZE x IMG_SIZE 리사이즈 좌표로 변환.'''
    w, h = original_size
    sx, sy = IMG_SIZE / w, IMG_SIZE / h
    return [round(box[0] * sx), round(box[1] * sy), round(box[2] * sx), round(box[3] * sy)]

show_sample_with_box(train_dataset, 0)
show_sample_with_box(test_dataset, 0)

## 3. 모델 구성 — ResNet50 + GAP + Dense(FC)

CAM을 얻으려면 모델 구조가 **Conv 특징맵 → Global Average Pooling(GAP) → 단일 Dense(FC) 레이어**로
끝나야 합니다. `torchvision.models.resnet50` 은 이미 `avgpool`(GAP) 다음에 `fc`(Dense) 레이어로
이어지는 구조이므로, ImageNet 사전학습 가중치를 불러오고 `fc`만 클래스 수(120)에 맞게 교체하면 됩니다.

CAM 수식: $\text{CAM}_c(x, y) = \sum_k w_k^c \cdot f_k(x, y)$
여기서 $f_k$ 는 마지막 conv block(`layer4`)의 $k$번째 채널 특징맵, $w_k^c$ 는 클래스 $c$에 대한
FC 레이어의 가중치입니다. 이 식이 성립하려면 GAP → FC 구조가 반드시 유지되어야 합니다.

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_dataset.classes))
model = model.to(device)

# 실습 편의를 위해 backbone은 고정하고 GAP 뒤의 FC(Dense) 레이어만 학습한다.
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True

print(model.fc)

## 4. 모델 학습

`fc` 레이어만 학습하는 linear-probe 방식으로 빠르게 수렴시킵니다. epoch별 train loss/accuracy를
기록해 학습이 안정적으로 수렴하는지 그래프로 확인합니다.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

history = {"loss": [], "acc": []}

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y, *_ in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    history["loss"].append(epoch_loss)
    history["acc"].append(epoch_acc)
    print(f"epoch={epoch+1}/{EPOCHS}  loss={epoch_loss:.4f}  train_acc={epoch_acc:.4f}")

torch.save(model.state_dict(), "stanford_dogs_resnet50_cam.pth")
print("모델 저장 완료: stanford_dogs_resnet50_cam.pth")

In [ ]:
# 학습 수렴 곡선 시각화 — 루브릭 3: 학습이 안정적으로 수렴하는지 확인
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(range(1, EPOCHS + 1), history["loss"], marker="o")
axes[0].set_title("Train Loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")

axes[1].plot(range(1, EPOCHS + 1), history["acc"], marker="o", color="darkorange")
axes[1].set_title("Train Accuracy")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
plt.tight_layout()
plt.show()

## 5. CAM (Class Activation Mapping) 구현

`layer4`(마지막 conv block)의 출력을 forward hook으로 저장해 두고, 예측 클래스에 해당하는
`fc.weight` 벡터로 가중합을 구해 class activation map을 얻습니다.

In [ ]:
class CAM:
    def __init__(self, model):
        self.model = model.eval()
        self.activation = None
        self.handle = model.layer4.register_forward_hook(self.hook)

    def hook(self, module, inputs, output):
        self.activation = output.detach()

    def remove(self):
        self.handle.remove()

    @torch.no_grad()
    def __call__(self, x):
        logits = self.model(x)
        pred = logits.argmax(1)
        weight = self.model.fc.weight[pred]                      # (B, C)
        heat = F.relu((weight[:, :, None, None] * self.activation).sum(1))
        heat = F.interpolate(heat[:, None], size=x.shape[-2:],
                              mode="bilinear", align_corners=False)[:, 0]
        return logits, pred, heat / (heat.amax((1, 2), keepdim=True) + 1e-8)

In [ ]:
def overlay_heatmap(img_np, heat, alpha=0.45, cmap="jet"):
    '''img_np: (H, W, 3) uint8, heat: (H, W) in [0, 1] -> overlay 이미지 반환'''
    colormap = plt.get_cmap(cmap)
    heat_rgb = (colormap(heat)[..., :3] * 255).astype(np.uint8)
    overlay = (heat_rgb * alpha + img_np * (1 - alpha)).astype(np.uint8)
    return overlay


def show_cam_result(img_np, heat, title):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(heat, cmap="jet"); axes[1].set_title(f"{title} heatmap"); axes[1].axis("off")
    axes[2].imshow(overlay_heatmap(img_np, heat)); axes[2].set_title(f"{title} overlay"); axes[2].axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
sample_idx = 0
image, label, box, size, path = test_dataset[sample_idx]
x = image.unsqueeze(0).to(device)
orig_img = unnormalize(image)

cam_engine = CAM(model)
logits, pred, heat = cam_engine(x)
cam_engine.remove()
heat_np = heat[0].cpu().numpy()

print("정답 클래스:", test_dataset.classes[label])
print("예측 클래스:", test_dataset.classes[pred.item()])
show_cam_result(orig_img, heat_np, "CAM")

## 6. Grad-CAM 구현

Grad-CAM은 GAP-FC 구조에 제약되지 않고 임의의 conv layer에 대해 계산할 수 있습니다.
관심 클래스에 대한 로짓의 gradient를 해당 레이어의 출력에 대해 backward하여
채널별 gradient의 GAP 값을 weight로 사용합니다.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model.eval()
        self.activation = None
        self.gradient = None
        self.h1 = model.layer4.register_forward_hook(self.fwd)
        self.h2 = model.layer4.register_full_backward_hook(self.bwd)

    def fwd(self, module, inputs, output):
        self.activation = output

    def bwd(self, module, grad_input, grad_output):
        self.gradient = grad_output[0]

    def remove(self):
        self.h1.remove()
        self.h2.remove()

    def __call__(self, x):
        # 백본 파라미터가 동결되어 있어도 중간 레이어까지 그래디언트를 전달하도록 강제
        x = x.requires_grad_(True)

        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)
        pred = logits.argmax(1)

        # Target Class Score 계산 후 역전파
        score = logits[torch.arange(x.size(0), device=x.device), pred].sum()
        score.backward()

        if self.gradient is None:
            raise RuntimeError("Gradient가 계산되지 않았습니다.")

        alpha = self.gradient.mean((2, 3), keepdim=True)
        heat = F.relu((alpha * self.activation).sum(1))
        heat = F.interpolate(heat[:, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]

        return logits.detach(), pred.detach(), heat.detach() / (heat.detach().amax((1, 2), keepdim=True) + 1e-8)

In [ ]:


# 입력 텐서 x의 연산 그래프 추적을 활성화
x = x.requires_grad_(True)

gradcam_engine = GradCAM(model)
logits_g, pred_g, heat_g = gradcam_engine(x)
gradcam_engine.remove()

heat_g_np = heat_g[0].cpu().numpy()

print("예측 클래스(Grad-CAM):", test_dataset.classes[pred_g.item()])
show_cam_result(orig_img, heat_g_np, "Grad-CAM")

## 7. CAM vs Grad-CAM 나란히 비교

같은 샘플에 대해 CAM과 Grad-CAM의 히트맵/오버레이를 나란히 배치해 두 방법이
물체의 주요 특징 위치(얼굴, 몸통 윤곽 등)를 얼마나 잘 짚어내는지 비교합니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for row, (name, heat_arr) in enumerate([("CAM", heat_np), ("Grad-CAM", heat_g_np)]):
    axes[row, 0].imshow(orig_img); axes[row, 0].set_title(f"{name}: Original"); axes[row, 0].axis("off")
    axes[row, 1].imshow(heat_arr, cmap="jet"); axes[row, 1].set_title(f"{name}: heatmap"); axes[row, 1].axis("off")
    axes[row, 2].imshow(overlay_heatmap(orig_img, heat_arr)); axes[row, 2].set_title(f"{name}: overlay"); axes[row, 2].axis("off")
plt.tight_layout()
plt.show()

## 8. 바운딩 박스 추출 및 IoU 계산

히트맵을 threshold로 이진화해 활성 영역의 최소/최대 좌표를 바운딩 박스로 변환하고,
GT 박스와의 IoU를 계산합니다.

In [ ]:
def mask_to_box(heat, threshold=0.5):
    ys, xs = np.where(heat >= threshold)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]


def iou(a, b):
    if a is None or b is None:
        return 0.0
    xa, ya, xb, yb = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xb - xa + 1) * max(0, yb - ya + 1)
    aa = (a[2] - a[0] + 1) * (a[3] - a[1] + 1)
    ab = (b[2] - b[0] + 1) * (b[3] - b[1] + 1)
    return inter / (aa + ab - inter + 1e-8)

In [ ]:
def draw_boxes(img_np, boxes, colors, labels):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img_np)
    for box, color, label in zip(boxes, colors, labels):
        if box is None:
            continue
        x0, y0, x1, y1 = box
        rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2,
                                  edgecolor=color, facecolor="none", label=label)
        ax.add_patch(rect)
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.show()


gt_box = map_box(box, size)
cam_box = mask_to_box(heat_np)
gradcam_box = mask_to_box(heat_g_np)

print("GT box       :", gt_box)
print("CAM box      :", cam_box, " IoU =", round(iou(cam_box, gt_box), 4))
print("Grad-CAM box :", gradcam_box, " IoU =", round(iou(gradcam_box, gt_box), 4))

draw_boxes(orig_img, [gt_box, cam_box, gradcam_box],
           ["lime", "red", "cyan"], ["GT", "CAM", "Grad-CAM"])
# 단일 이미지 결과 확인
gradcam_engine = GradCAM(model)
logits_g, pred_g, heat_g = gradcam_engine(x_in)
gradcam_engine.remove()
heat_g_np = heat_g[0].cpu().numpy()

print("예측 클래스(Grad-CAM):", test_dataset.classes[pred_g.item()])
show_cam_result(orig_img, heat_g_np, "Grad-CAM")

## 9. 테스트셋 전체 정량 평가

테스트셋 전체에 대해 분류 정확도, 평균 IoU, IoU ≥ 0.5 비율을 CAM과 Grad-CAM 각각 계산해
두 방법의 localization 성능을 정량적으로 비교합니다.

In [ ]:
def evaluate(engine_cls, name):
    engine = engine_cls(model)
    ious, correct = [], 0
    for x_i, y_i, box_i, size_i, _ in test_dataset:
        x_in = x_i.unsqueeze(0).to(device)
        if name == "Grad-CAM":
            with torch.enable_grad():
                _, pred_i, heat_i = engine(x_in)
        else:
            with torch.no_grad():
                _, pred_i, heat_i = engine(x_in)
        correct += int(pred_i.item() == y_i)
        pred_box = mask_to_box(heat_i[0].cpu().numpy())
        ious.append(iou(pred_box, map_box(box_i, size_i)))
    engine.remove()
    ious = np.array(ious)
    return {
        "accuracy": correct / len(test_dataset),
        "mean_iou": float(ious.mean()),
        "iou_ge_0.5": float((ious >= 0.5).mean()),
    }

# CAM 및 Grad-CAM 평가 진행
results = {
    "CAM": evaluate(CAM, "CAM"),
    "Grad-CAM": evaluate(GradCAM, "Grad-CAM"),
}

for name, metrics in results.items():
    print(f"{name}: accuracy={metrics['accuracy']:.4f}  "
          f"mean_iou={metrics['mean_iou']:.4f}  "
          f"iou>=0.5 rate={metrics['iou_ge_0.5']:.4f}")

In [ ]:
# 결과 비교 막대그래프
metric_names = ["accuracy", "mean_iou", "iou_ge_0.5"]
x_pos = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
for i, name in enumerate(["CAM", "Grad-CAM"]):
    values = [results[name][m] for m in metric_names]
    ax.bar(x_pos + i * width, values, width, label=name)

ax.set_xticks(x_pos + width / 2)
ax.set_xticklabels(["Accuracy", "Mean IoU", "IoU>=0.5 rate"])
ax.set_ylim(0, 1)
ax.set_title("CAM vs Grad-CAM — Localization 성능 비교")
ax.legend()
plt.tight_layout()
plt.show()

## 10. 여러 샘플에 대한 CAM/Grad-CAM 비교 (정성적 분석)

정량 지표 외에, 실제 여러 샘플에서 두 방법이 만든 히트맵과 바운딩 박스를 나란히 살펴보며
어떤 상황에서 각 방법이 더 정확하게(혹은 부정확하게) localization 하는지 정성적으로 확인합니다.

In [ ]:
def compare_samples(dataset, indices):
    cam_engine = CAM(model)
    grad_engine = GradCAM(model)
    for idx in indices:
        image, label, box_i, size_i, _ = dataset[idx]
        x_in = image.unsqueeze(0).to(device)
        img_np = unnormalize(image)

        with torch.no_grad():
            _, pred_c, heat_c = cam_engine(x_in)
        with torch.enable_grad():
            _, pred_g, heat_g_ = grad_engine(x_in)

        heat_c_np = heat_c[0].cpu().numpy()
        heat_g_np_ = heat_g_[0].detach().cpu().numpy()
        gt = map_box(box_i, size_i)
        cam_b = mask_to_box(heat_c_np)
        grad_b = mask_to_box(heat_g_np_)

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_np); axes[0].set_title(f"GT: {dataset.classes[label]}"); axes[0].axis("off")
        axes[1].imshow(overlay_heatmap(img_np, heat_c_np))
        axes[1].set_title(f"CAM (IoU={iou(cam_b, gt):.2f})"); axes[1].axis("off")
        axes[2].imshow(overlay_heatmap(img_np, heat_g_np_))
        axes[2].set_title(f"Grad-CAM (IoU={iou(grad_b, gt):.2f})"); axes[2].axis("off")
        plt.tight_layout()
        plt.show()
    cam_engine.remove()
    grad_engine.remove()

compare_samples(test_dataset, indices=[1, 2, 3])

## 11. 결과 분석 및 결론

아래 항목을 채우며 CAM과 Grad-CAM의 결과를 비교 분석해 봅시다.

- **분류 성능**: CAM과 Grad-CAM은 동일한 분류기의 예측을 사용하므로 accuracy는 이론상 동일해야 합니다.
  실제로 위 결과에서 두 값이 같은지 확인하세요.
- **Localization 성능**: `mean_iou`, `iou>=0.5 rate` 지표를 비교했을 때 어떤 방법이 더 우수했나요?
  구조적 제약(GAP→FC) 없이 임의의 레이어에 적용 가능한 Grad-CAM이 유연성 면에서 어떤 이점이 있었는지
  서술해 보세요.
- **정성적 관찰**: 9번 섹션에서 살펴본 샘플들에서 히트맵이 개(dog)의 얼굴/몸통 중 어느 부분에
  더 집중되었는지, 배경(background)에 잘못 반응한 사례는 없었는지 기록해 보세요.
- **한계 및 개선 방향**: 히트맵 이진화 threshold(0.5)를 바꾸면 IoU가 어떻게 달라지는지,
  더 많은 학습 데이터/에폭으로 학습했을 때 localization 성능이 개선되는지 실험해 보는 것도
  좋은 후속 분석이 될 수 있습니다.

> (자유롭게 본인의 실험 결과와 분석 내용을 이 마크다운 셀에 작성하세요.)